# Appendix A: Security Command Reference

This appendix provides a concise reference for command-line tools used throughout the book.
All commands should be run only on systems you own or are explicitly authorised to test.

---

## Network Scanning and Enumeration

### Nmap

```bash
# SYN scan all ports with service/version and OS detection
nmap -sS -sV -O -p- -T4 <target>

# Aggressive scan with NSE default scripts
nmap -A -p 80,443,22,21,3389 <target>

# Run specific NSE scripts
nmap --script smb-enum-shares,smb-enum-users <target>
nmap --script ssl-cert,ssl-enum-ciphers -p 443 <target>

# Save output in all formats
nmap -sV -oA scan_results <target>

# UDP scan (top 100 UDP ports)
nmap -sU --top-ports 100 <target>

# Decoy scan (blend with decoys)
nmap -sS -D RND:10 <target>
```

### DNS Enumeration

```bash
# Basic record lookups
dig A target.com
dig MX target.com
dig TXT target.com
dig NS target.com

# Zone transfer attempt
dig AXFR @ns1.target.com target.com

# DNSSEC validation
dig +dnssec A target.com

# Reverse lookup
dig -x 203.0.113.10
```

### Web Enumeration

```bash
# Directory brute-force
gobuster dir -u http://target.com -w /usr/share/seclists/Discovery/Web-Content/common.txt
ffuf -u http://target.com/FUZZ -w wordlist.txt -mc 200,301,302

# Parameter fuzzing
ffuf -u "http://target.com/page?FUZZ=value" -w params.txt

# Virtual host enumeration
gobuster vhost -u http://target.com -w subdomains.txt
```

---

## Password and Credential Tools

```bash
# Offline hash cracking
hashcat -m 0 hashes.txt rockyou.txt              # MD5
hashcat -m 1000 hashes.txt rockyou.txt            # NTLM
hashcat -m 3200 hashes.txt rockyou.txt            # bcrypt

# Online password spray (authorised testing only)
crackmapexec smb <target_subnet> -u users.txt -p 'Password123' --continue-on-success

# Extract hashes from memory (Windows; requires admin)
# mimikatz: sekurlsa::logonpasswords
```

---

## Forensics

```bash
# Create forensic image
dd if=/dev/sdb of=disk.img bs=4M status=progress conv=sync,noerror
dcfldd if=/dev/sdb of=disk.img hash=sha256 hashlog=hash.log

# Hash verification
sha256sum disk.img

# File type identification
file unknown_file
xxd unknown_file | head -4    # View magic bytes

# String extraction
strings -a -n 8 binary_file > strings.txt

# Entropy estimation (via binwalk)
binwalk -E file.bin

# File carving
binwalk -e archive.bin
foremost -i disk.img -o recovered/

# Memory forensics
volatility3 -f memory.dmp windows.pslist
volatility3 -f memory.dmp windows.netscan
volatility3 -f memory.dmp windows.malfind
```

---

## Network Analysis

```bash
# Capture traffic
tcpdump -i eth0 -w capture.pcap
tcpdump -i eth0 port 80 -A      # HTTP in ASCII

# Filter in Wireshark / tshark
tshark -r capture.pcap -Y "http.request"
tshark -r capture.pcap -Y "dns" -T fields -e dns.qry.name

# NetFlow (if yaf installed)
yaf --in=eth0 --out=flow.ipfix
```

---

## Cryptography

```bash
# Generate keys and certificates
openssl genrsa -out private.key 4096
openssl req -new -x509 -key private.key -out cert.pem -days 365
openssl s_client -connect target.com:443 -showcerts

# Hashing
echo -n "hello" | sha256sum
openssl dgst -sha256 file.txt

# File encryption with AES-256-GCM
openssl enc -aes-256-gcm -salt -in plaintext.txt -out encrypted.bin
openssl enc -d -aes-256-gcm -in encrypted.bin -out decrypted.txt

# Check TLS configuration
nmap --script ssl-enum-ciphers -p 443 target.com
testssl.sh target.com
```

---

## Python One-Liners

```python
# Base64 encode/decode
python3 -c "import base64; print(base64.b64encode(b'hello').decode())"
python3 -c "import base64; print(base64.b64decode('aGVsbG8=').decode())"

# SHA-256 hash
python3 -c "import hashlib; print(hashlib.sha256(b'hello').hexdigest())"

# Simple HTTP server
python3 -m http.server 8080

# Quick port check
python3 -c "import socket; s=socket.socket(); s.settimeout(1); print(s.connect_ex(('target.com',443)))"
```
